In [1]:

# #! echo $PWD
! pip install ../utility_functions/
# #!pip install ~/fwiViz/utility_functions/

Processing /home/jovyan/fwiVis/utility_functions
  Preparing metadata (setup.py) ... done
  Created wheel for fwiVis: filename=fwiVis-0.1-py3-none-any.whl size=17985 sha256=a941d6460d322aa045fcc980e0b661a5ec88e01255da00aa67574d0cf15171be
  Stored in directory: /tmp/pip-ephem-wheel-cache-rg1htvuo/wheels/79/0f/3d/08c18473dd7e0fb915900e6b4f13b81f1fa84371f9bf24d864
Successfully built fwiVis


In [3]:
import fwiVis.fwiVis as fv
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain
from bs4 import BeautifulSoup # I mamba installed bs4
import requests

from datetime import date

In [3]:



# def make_df_of_fires(year = "2023", path_region = "Quebec_PostHoc", custom_path = "/home/jovyan/fireatlast_nrt/fireatlas/data/FEDSoutput-v3/"):

#     diroutdata = custom_path
#     spath = os.path.join(diroutdata, path_region, str(year), "Largefire")
#     fnms = [f for f in os.listdir(spath)]
#     print(fnms)
#     #fnms = fnms.sort()
#     tmp_ids = pd.DataFrame(fnms, columns=["ids"])
#     #tmp_ids = tmp_ids[~tmp_ids.ids.str.contains(".")]
#     print(tmp_ids)
#     tmp_ids = tmp_ids.ids.unique()
#     print(f'{len(tmp_ids)} unique ID found')

#     print("Reading in IDS")
#     ### reading in the ids
#     fires = pd.DataFrame()
#     for n,i in enumerate(tmp_ids, start = 0):
#         try:
#             foo = fv.load_large_fire(i, year = year, path_region= path_region, layer = "perimeter",  s3_path = False, custom_path = custom_path)
#             foo["fireID"] = str(i)
#         except Exception as e:
#             print("Error at ID: ",i)
#             print(e)
#             continue
#         ## Extract the period between 
#         fires = pd.concat([fires, foo])
#     return (fires)
#         #print(fires)
#             #fr_pd = pd.DataFrame(fires, columns=["lat", "lon", "farea", "data_source"])
#         #fires.to_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/"+"20_days_fire_stats_only_718270-99999_" +min_t + max_t + path_region + FWI_source + str(date.today().strftime("%Y%m%d"))+  ".csv")

In [4]:
# fires = make_df_of_fires()

In [4]:
## but wait, maybe Julia made a function for combined lf anyway


lf = gpd.read_file("/home/jovyan/fireatlast_nrt/fireatlas/data/FEDSoutput-v3/Quebec_V3/2023/CombinedLargefire/20230915PM/lf_perimeter.fgb")

In [5]:
len(lf.fireID.unique())

204

In [6]:
def get_largest_perimeter(df):
    fireID = df.fireID
    df = df[(np.round(df.farea, 4) == np.round(df.farea.max(), 4)) & (df.t == df.t.max())]
    if(len(df) == 0):
        print(f"Warning! dropping ID {fireID} because max t didn't match max farea.")
    return(df)

In [8]:
#new_lf = lf.groupby("fireID").apply(get_largest_perimeter,  include_groups = True )

In [9]:
#new_lf.explore()

In [10]:
# print(len(lf.fireID.unique()))
# len(new_lf.fireID.unique())

In [11]:
# new_lf = new_lf.to_crs("4326")
# new_lf["lat_centroid"] = new_lf.geometry.centroid.y
# new_lf["lon_centroid"] = new_lf.geometry.centroid.x

# new_lf.columns

In [12]:
# new_lf.fireID = new_lf.fireID.astype("int")
# new_lf.mergeid = new_lf.mergeid.astype("int")

In [13]:
lf.fireID = lf.fireID.astype("int")
lf.mergeid = lf.mergeid.astype("int")

In [14]:
# import datetime
# now = date.today()
# new_lf.fireID = new_lf.fireID.astype('int')
# new_lf.mergeid = new_lf.mergeid.astype('int')
# new_lf.lon_centroid = new_lf.centroid.x
# new_lf.lat_centroid = new_lf.centroid.y
# new_lf = new_lf.drop(columns = "geometry")

# new_lf.to_csv("~/fwiVis/notebooks/data/Quebec_v3_last_perimeters_as_of_" + str(now.year)+str(now.month)+str(now.day)+".csv")

In [15]:
# print(len(lf[~lf.fireID.isin(new_lf.fireID.unique())].fireID.unique()))
# weird_ids = lf[~lf.fireID.isin(new_lf.fireID.unique())].fireID.unique()
# print(weird_ids)

In [45]:
def listFD(url, ext=''):
    page = requests.get(url).text
    #print(page)
    soup = BeautifulSoup(page, 'html.parser')
    return [url + '/' + node.get('href') for node in soup.find_all('a') if node.get('href').endswith(ext)]

## https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.247.biggestFires/
# https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.216.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/
# https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3.Radius.50.km.176.biggestFires/
# 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.247.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/
# 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3.Radius.50.km.176.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/MAX/chicletDataNoSmoothing/'
def get_nccs_url(pattern, url = 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3.Radius.50.km.204.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/MAX/chicletDataNoSmoothing/', ext = 'csv', pattern2 = "FWI.raw"):    

#url = 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecAllFires.Radius.25.km.216.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/chicletDataNoSmoothing/'
#ext = 'csv'
    file_list = []
    for file in listFD(url, ext):
        file_list.append(file)

    try_pd = pd.DataFrame(file_list, columns= ["urls"])
    size = try_pd[try_pd.urls.str.contains(pattern)].urls.values.size
    if(pattern2 is not None):
        #print(f"Searching by second pattern {pattern2}")
        size = try_pd[(try_pd.urls.str.contains(pattern)) & (try_pd.urls.str.contains(pattern2))].urls.values.size
        print(size)
    if(size == 0):
        print("No matches found to pattern. Returning None.")
        return(None)
    if(size >= 2):
        print("Multiple matches found:")
        #print(try_pd[try_pd.urls.str.contains(pattern)].urls.values)
        print(try_pd[try_pd.urls.str.contains(pattern) & try_pd.urls.str.contains(pattern2)].urls.values)
        raise ValueError()
    url = try_pd[try_pd.urls.str.contains(pattern)].urls.values[0]
    if(pattern2 is not None):
        url = try_pd[try_pd.urls.str.contains(pattern) & try_pd.urls.str.contains(pattern2)].urls.values[0]
    return(url)

def get_gridded_fwi(fireID):
    
    # Get the URL for the file
    fireID = str(fireID)
    #pattern = "FWI." + fireID
    pattern = "\." + fireID + "\_Lat"
    url = get_nccs_url(pattern = pattern, pattern2 = pattern2)
    print(url)
    if(url is not None):
        
        # Get the DF
        grid_FWI = pd.read_csv(url)
        # Change names
        grid_FWI = grid_FWI.rename(columns={'INITDATE': 't', 
                                 "0":"FWI",
                                 "1":"FWI_lead_1",
                                 "2":"FWI_lead_2",
                                 "3":"FWI_lead_3",
                                 "4":"FWI_lead_4",
                                 "5":"FWI_lead_5",
                                 "6":"FWI_lead_6",
                                 "7":"FWI_lead_7",
                                 "8":"FWI_lead_8"
                                })
        # Change dates
        grid_FWI.t = grid_FWI.t.astype("datetime64[ns]").dt.strftime('%Y-%m-%d 12:00:00')

        # return
        return(grid_FWI)
    else:
        return(None)

# 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3.Radius.50.km.204.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/MEAN/chicletDataNoSmoothing/
def get_gridded_met(fireID, met_name = "FWI", pattern2 = "FWI.raw", url = 'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3.Radius.50.km.204.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/MAX/chicletDataNoSmoothing/',  pattern3 = ""):
    
    # Get the URL for the file
    fireID = str(fireID)
    #pattern = "ISI." + fireID
    pattern = "\." + fireID + "\_Lat"
    url = get_nccs_url(pattern = pattern, pattern2 = pattern2, url = url)
    #print(url)
    if(url is not None):
        
        # Get the DF
        grid = pd.read_csv(url)
        # Change names
        grid = grid.rename(columns={'INITDATE': 't', 
                                 "GEOS-5.IMERGEARLY":f"GEOS-5.IMERGEARLY{pattern3}",
                                 "0":f"{met_name}",
                                 "1":f"{met_name}_lead_1",
                                 "2":f"{met_name}_lead_2",
                                 "3":f"{met_name}_lead_3",
                                 "4":f"{met_name}_lead_4",
                                 "5":f"{met_name}_lead_5",
                                 "6":f"{met_name}_lead_6",
                                 "7":f"{met_name}_lead_7",
                                 "8":f"{met_name}_lead_8"
                                })
        #print(grid)
        # Change dates
        grid.t = grid.t.astype("datetime64[ns]").dt.strftime('%Y-%m-%d 12:00:00')

        # return
        return(grid)
    else:
        print("Url is none.")
        print(url)
        return(None)



<>:44: SyntaxWarning: invalid escape sequence '\.'
<>:44: SyntaxWarning: invalid escape sequence '\_'
<>:77: SyntaxWarning: invalid escape sequence '\.'
<>:77: SyntaxWarning: invalid escape sequence '\_'
<>:44: SyntaxWarning: invalid escape sequence '\.'
<>:44: SyntaxWarning: invalid escape sequence '\_'
<>:77: SyntaxWarning: invalid escape sequence '\.'
<>:77: SyntaxWarning: invalid escape sequence '\_'
/tmp/ipykernel_397/2874259750.py:44: SyntaxWarning: invalid escape sequence '\.'
  pattern = "\." + fireID + "\_Lat"
/tmp/ipykernel_397/2874259750.py:44: SyntaxWarning: invalid escape sequence '\_'
  pattern = "\." + fireID + "\_Lat"
/tmp/ipykernel_397/2874259750.py:77: SyntaxWarning: invalid escape sequence '\.'
  pattern = "\." + fireID + "\_Lat"
/tmp/ipykernel_397/2874259750.py:77: SyntaxWarning: invalid escape sequence '\_'
  pattern = "\." + fireID + "\_Lat"


In [46]:
# lf.to_csv("~/fwiVis/notebooks/data/Quebec_v3_perimeters_as_of_" + str(now.year)+str(now.month)+str(now.day)+".csv")

In [47]:
smol = lf[lf.fireID == lf.fireID.iloc[0]]
fireID = smol.fireID.iloc[0]
fwi = get_gridded_met(str(fireID))

1


In [48]:
#fwi.merge(smol, on = ["t"], how = "outer").n_newpixels.unique()
fwi = get_gridded_met(str("226"))

1


In [49]:
def merge_gridded_FWI_data(df, FWI_subset_t = False, met_vars = ["FWI", "ISI", "BUI"], 
                           met_urls = ['https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3MoreIndices.Radius.50.km.204.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/MAX/chicletDataNoSmoothing/',
                                       'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3MoreIndices.Radius.50.km.204.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/MAX/chicletDataNoSmoothing/', 
                                       'https://portal.nccs.nasa.gov/datashare/GlobalFWI/ForecastFWIEXPERIMENTAL/QuebecFEDSv3MoreIndices.Radius.50.km.204.biggestFires/GEOS-5/GEOS-5.IMERGEARLY/MAX/chicletDataNoSmoothing/' ],
                           met_pattern2 = ["FWI.raw", "ISI.raw", "BUI.raw"], met_pattern3 = ["", "_ISI", "_BUI"]):
    fireID = df.fireID.iloc[0]
    df.t = df.t.astype("datetime64[ns]")
    result_df = df.copy()  # Create a copy of the original dataframe
    
    for i, m in enumerate(met_vars):
        current_fwi = get_gridded_met(str(fireID), met_name = met_vars[i], 
                                    pattern2 = met_pattern2[i], url = met_urls[i], pattern3 = met_pattern3[i])
        #print(f'{met_pattern2[i]}')
        #print(f'{met_vars[i]}')
       # print(current_fwi[f'{met_vars[i]}'].iloc[203])
        #print(f'{met_vars[i]}')
        if(current_fwi is None):
            print(f"Warning:{met_vars[i]} in {fireID} was not extracted sucessfully.")
            print({met_urls[i]})
            return None
        else:
            current_fwi.t = current_fwi.t.astype("datetime64[ns]")
            current_fwi.loc[:, "fireID"] = fireID
            
            if(FWI_subset_t):
                current_fwi = current_fwi[current_fwi.t.astype("datetime64[ns]") >= result_df.t.astype("datetime64[ns]").min()]
                current_fwi = current_fwi[current_fwi.t.astype("datetime64[ns]") <= result_df.t.astype("datetime64[ns]").max()]
        
            result_df = result_df.merge(current_fwi, on = ["fireID", "t"], how = "outer")
    
    return result_df


In [51]:
col_names = lf.columns

lf.fireID = lf.fireID.astype("int")
lf.fireID = lf.fireID.astype("str")


merged_lf = lf.groupby("fireID")[col_names].apply(merge_gridded_FWI_data).reset_index(drop=True) #include_groups=True ## This rearranges the columns so that fireID isn't first. 
#test = smol.groupby("fireID")[col_names].apply(merge_gridded_FWI_data).reset_index() #include_groups=True
merged_lf 

#merged_lf.BUI.dropna() == merged_lf.FWI.dropna()

1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


,mergeid,ftype,n_pixels,n_newpixels,farea,fperim,flinelen,duration,pixden,meanFRP,...,GEOS-5.IMERGEARLY_BUI,BUI,BUI_lead_1,BUI_lead_2,BUI_lead_3,BUI_lead_4,BUI_lead_5,BUI_lead_6,BUI_lead_7,BUI_lead_8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157691,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157692,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157693,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
157694,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
# col_names = lf.columns

# lf.fireID = lf.fireID.astype("int")
# lf.fireID = lf.fireID.astype("str")

# tmp = lf[lf.fireID == '226']
# merged_lf = merge_gridded_FWI_data(tmp) #include_groups=True ## This rearranges the columns so that fireID isn't first. 
# #test = smol.groupby("fireID")[col_names].apply(merge_gridded_FWI_data).reset_index() #include_groups=True
# merged_lf 

# #merged_lf.BUI.dropna() == merged_lf.FWI.dropna()

/srv/conda/envs/notebook/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


1
1
1


,mergeid,ftype,n_pixels,n_newpixels,farea,fperim,flinelen,duration,pixden,meanFRP,...,GEOS-5.IMERGEARLY_BUI,BUI,BUI_lead_1,BUI_lead_2,BUI_lead_3,BUI_lead_4,BUI_lead_5,BUI_lead_6,BUI_lead_7,BUI_lead_8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
832,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
833,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
835,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# col_names = lf.columns

# lf.fireID = lf.fireID.astype("int")
# lf.fireID = lf.fireID.astype("str")


# merged_lf = lf.groupby("fireID")[col_names].apply(merge_gridded_FWI_data).reset_index(drop=True) #include_groups=True ## This rearranges the columns so that fireID isn't first. 
# #test = smol.groupby("fireID")[col_names].apply(merge_gridded_FWI_data).reset_index() #include_groups=True
# merged_lf 

In [52]:
import datetime
now = date.today()

merged_lf.to_csv("~/fwiVis/notebooks/data/Quebec_v3_full_data_perimeters"+ "MAX_fwi_with_ISI_and_BUI" + "ful_timeseries_" + str(now.year)+str(now.month)+str(now.day)+".csv")

In [24]:
merged_lf['GEOS-5.IMERGEARLY'].unique()

array([           nan, 4.18151855e+00, 4.58382750e+00, 1.32272422e-01,
       1.00814700e+00, 1.14874291e+00, 4.36931992e+00, 5.03310299e+00,
       6.73083115e+00, 8.09321594e+00, 8.35452652e+00, 9.43735027e+00,
       9.98350143e+00, 1.46826935e+01, 7.72254133e+00, 6.36088896e+00,
       8.49648571e+00, 1.04301088e-01, 3.31574082e+00, 4.20471668e+00,
       4.42501259e+00, 8.98884535e-01, 9.13744390e-01, 3.42909050e+00,
       8.14196396e+00, 1.22817268e+01, 1.18143663e+01, 1.30960722e+01,
       1.67687263e+01, 1.76056156e+01, 1.15633783e+01, 1.06273909e+01,
       1.36381779e+01, 7.62161589e+00, 1.08729610e+01, 1.06922922e+01,
       1.26898403e+01, 9.99564457e+00, 1.23744688e+01, 7.51217842e+00,
       7.72508812e+00, 5.04784212e-02, 3.30954354e-05, 1.39104620e-01,
       2.45117068e-01, 6.40013397e-01, 2.54480314e+00, 6.01722145e+00,
       9.29628658e+00, 1.02057085e+01, 1.36652498e+01, 1.43177271e+01,
       1.40790596e+01, 1.20598087e+01, 3.12030888e+00, 6.80023015e-01,
      

In [25]:
merged_lf[merged_lf.fireID == '1003'].n_pixels.unique()

array([], dtype=float64)

In [26]:
merged_lf.to_csv("~/fwiVis/notebooks/data/Quebec_v3_full_data_perimeters"+ "MAX_fwi_" + "ful_timeseries_" + str(now.year)+str(now.month)+str(now.day)+".csv")

In [27]:
test = pd.read_csv("~/fwiVis/notebooks/data/Quebec_v3_full_data_perimeters"+ "MAX_fwi_" + "ful_timeseries_" + str(now.year)+str(now.month)+str(now.day)+".csv")

In [28]:
test['GEOS-5.IMERGEARLY'].unique()

array([           nan, 4.18151855e+00, 4.58382750e+00, 1.32272422e-01,
       1.00814700e+00, 1.14874291e+00, 4.36931992e+00, 5.03310299e+00,
       6.73083115e+00, 8.09321594e+00, 8.35452652e+00, 9.43735027e+00,
       9.98350143e+00, 1.46826935e+01, 7.72254133e+00, 6.36088896e+00,
       8.49648571e+00, 1.04301088e-01, 3.31574082e+00, 4.20471668e+00,
       4.42501259e+00, 8.98884535e-01, 9.13744390e-01, 3.42909050e+00,
       8.14196396e+00, 1.22817268e+01, 1.18143663e+01, 1.30960722e+01,
       1.67687263e+01, 1.76056156e+01, 1.15633783e+01, 1.06273909e+01,
       1.36381779e+01, 7.62161589e+00, 1.08729610e+01, 1.06922922e+01,
       1.26898403e+01, 9.99564457e+00, 1.23744688e+01, 7.51217842e+00,
       7.72508812e+00, 5.04784212e-02, 3.30954354e-05, 1.39104620e-01,
       2.45117068e-01, 6.40013397e-01, 2.54480314e+00, 6.01722145e+00,
       9.29628658e+00, 1.02057085e+01, 1.36652498e+01, 1.43177271e+01,
       1.40790596e+01, 1.20598087e+01, 3.12030888e+00, 6.80023015e-01,
      

In [29]:
merged_lf[merged_lf.fireID == '512'].t.min()

NaT

In [30]:
merged_lf

,mergeid,ftype,n_pixels,n_newpixels,farea,fperim,flinelen,duration,pixden,meanFRP,...,GEOS-5.IMERGEARLY,BUI,BUI_lead_1,BUI_lead_2,BUI_lead_3,BUI_lead_4,BUI_lead_5,BUI_lead_6,BUI_lead_7,BUI_lead_8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
721,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
722,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
723,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
merged_lf.columns

Index(['mergeid', 'ftype', 'n_pixels', 'n_newpixels', 'farea', 'fperim',
       'flinelen', 'duration', 'pixden', 'meanFRP', 't', 't_st', 't_ed',
       'fireID', 'isignition', 't_inactive', 'isactive', 'isdead',
       'mayreactivate', 'geom_counts', 'low_confidence_grouping', 'region',
       'primarykey', 'geometry', 'GEOS-5.IMERGEARLY_x', 'FWI', 'FWI_lead_1',
       'FWI_lead_2', 'FWI_lead_3', 'FWI_lead_4', 'FWI_lead_5', 'FWI_lead_6',
       'FWI_lead_7', 'FWI_lead_8', 'GEOS-5.IMERGEARLY_y', 'ISI', 'ISI_lead_1',
       'ISI_lead_2', 'ISI_lead_3', 'ISI_lead_4', 'ISI_lead_5', 'ISI_lead_6',
       'ISI_lead_7', 'ISI_lead_8', 'GEOS-5.IMERGEARLY', 'BUI', 'BUI_lead_1',
       'BUI_lead_2', 'BUI_lead_3', 'BUI_lead_4', 'BUI_lead_5', 'BUI_lead_6',
       'BUI_lead_7', 'BUI_lead_8'],
      dtype='object')